# Step 1 — isolate and save the four horizon datasets

This notebook performs shared transformations once before horizon isolation: `published_at` becomes an Asia/Colombo `publish_time_bucket`, `ch_subs_at_publish` becomes a fixed `subscriber_tier`, clean pre-publication channel statistics produce `ch_avg_views_per_video_at_publish`, and topic URLs are canonicalised into 18 binary topic indicators plus a missing indicator. Backfilled channel averages are masked. The raw topic-combination string is not exported. It then filters day `X` with `eligible`, `dX_usable`, and non-missing `dX_views`, removes the horizon usability flag, and retains only that horizon's view target.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

HORIZONS = (7, 14, 21, 30)

project_root = Path.cwd()
if not (project_root / "Dataset").exists():
    project_root = project_root.parent

source_path = project_root / "Dataset" / "viewcastlk_training_table.csv"
output_dir = project_root / "Dataset" / "model_horizon_datasets"
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(source_path, low_memory=False)
print(f"Loaded {len(df):,} source rows")

Loaded 64,515 source rows


In [2]:
from urllib.parse import unquote, urlparse

# Engineer shared fields once on the master table before creating any horizon dataset.
published_at_slt = pd.to_datetime(
    df["published_at"], errors="coerce", utc=True
).dt.tz_convert("Asia/Colombo")
publish_hour_slt_from_timestamp = published_at_slt.dt.hour

df["publish_time_bucket"] = pd.cut(
    publish_hour_slt_from_timestamp,
    bins=[-0.001, 5.999, 14.999, 20.999, 23.999],
    labels=["early_morning", "morning_afternoon", "evening", "late_night"],
    include_lowest=True,
).astype("string").fillna("unknown")

assert df["publish_time_bucket"].notna().all()
assert set(df["publish_time_bucket"]).issubset({
    "early_morning", "morning_afternoon", "evening", "late_night", "unknown"
})

# Fixed, deployment-safe subscriber bands. These boundaries do not depend on
# target values or on a particular train/test split.
SUBSCRIBER_TIER_ORDER = (
    "under_1k", "1k_to_10k", "10k_to_100k",
    "100k_to_250k", "250k_to_500k", "500k_to_1m",
    "1m_plus", "missing",
)
subscriber_count = pd.to_numeric(df["ch_subs_at_publish"], errors="coerce")
df["subscriber_tier"] = pd.cut(
    subscriber_count,
    bins=[
        float("-inf"), 999, 9_999, 99_999,
        249_999, 499_999, 999_999, float("inf"),
    ],
    labels=SUBSCRIBER_TIER_ORDER[:-1],
    include_lowest=True,
).astype("string").fillna("missing")
assert df["subscriber_tier"].notna().all()
assert set(df["subscriber_tier"]).issubset(SUBSCRIBER_TIER_ORDER)
assert (df.loc[subscriber_count < 1_000, "subscriber_tier"] == "under_1k").all()
assert (df.loc[subscriber_count >= 1_000_000, "subscriber_tier"] == "1m_plus").all()

# Historical channel average, retained only when the channel snapshot is truly
# from at or before publication. Backfilled post-publication stats are masked
# to prevent target contamination.
channel_views = pd.to_numeric(df["ch_views_at_publish"], errors="coerce")
channel_videos = pd.to_numeric(
    df["ch_videos_at_publish"], errors="coerce"
).where(lambda values: values > 0)
channel_stats_backfilled = (
    df["channel_stats_backfilled"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin({"true", "1", "yes"})
)
df["ch_avg_views_per_video_at_publish"] = (
    channel_views / channel_videos
).replace([float("inf"), float("-inf")], float("nan")).mask(
    channel_stats_backfilled
)
assert df.loc[channel_stats_backfilled, "ch_avg_views_per_video_at_publish"].isna().all()
assert not df["ch_avg_views_per_video_at_publish"].dropna().isin([float("inf"), float("-inf")]).any()


TOPIC_CANONICAL_GROUPS = {
    "Music": {
        "Christian music", "Classical music", "Electronic music", "Hip hop music",
        "Music", "Music of Asia", "Music of Latin America", "Pop music",
        "Rock music", "Soul music",
    },
    "Gaming": {
        "Action game", "Action-adventure game", "Casual game", "Puzzle video game",
        "Racing video game", "Role-playing video game", "Simulation video game",
        "Sports game", "Strategy video game", "Video game culture",
    },
    "Sports": {
        "Association football", "Boxing", "Cricket", "Motorsport", "Sport", "Volleyball",
    },
    "Entertainment": {
        "Entertainment", "Film", "Performing arts", "Television program",
    },
    "Health": {"Health", "Physical fitness"},
    "Lifestyle": {"Lifestyle (sociology)"},
    "Politics": {"Politics", "Military"},
    "Knowledge": {"Knowledge", "Business"},
}
TOPIC_CANONICAL_LOOKUP = {
    original: canonical
    for canonical, originals in TOPIC_CANONICAL_GROUPS.items()
    for original in originals
}
TOPIC_PARENT_CHILDREN = {
    "Society": {"Politics", "Religion"},
    "Lifestyle": {"Tourism", "Vehicle", "Hobby", "Food", "Fashion", "Pet", "Technology", "Health"},
    "Entertainment": {"Humour"},
}


def extract_topic_labels(value):
    """Extract, canonicalise, deduplicate, and sort pipe-separated topic URLs."""
    if pd.isna(value):
        return pd.NA
    labels = []
    for item in str(value).split("|"):
        item = item.strip()
        if not item:
            continue
        path = urlparse(item).path
        label = unquote(path.rsplit("/", 1)[-1]).replace("_", " ").strip()
        label = TOPIC_CANONICAL_LOOKUP.get(label, label)
        if label and label not in labels:
            labels.append(label)
    label_set = set(labels)
    for parent, children in TOPIC_PARENT_CHILDREN.items():
        if parent in label_set and label_set.intersection(children):
            label_set.remove(parent)
    return "|".join(sorted(label_set, key=str.casefold)) if label_set else pd.NA


df["topic_categories"] = df["topic_categories"].map(extract_topic_labels)
assert extract_topic_labels("https://en.wikipedia.org/wiki/Society") == "Society"
assert extract_topic_labels("https://en.wikipedia.org/wiki/Society|https://en.wikipedia.org/wiki/Politics") == "Politics"
assert extract_topic_labels("https://en.wikipedia.org/wiki/Lifestyle_(sociology)|https://en.wikipedia.org/wiki/Music|https://en.wikipedia.org/wiki/Music_of_Asia|https://en.wikipedia.org/wiki/Pop_music") == "Lifestyle|Music"
assert extract_topic_labels("https://en.wikipedia.org/wiki/Lifestyle_(sociology)|https://en.wikipedia.org/wiki/Tourism") == "Tourism"
assert extract_topic_labels("https://en.wikipedia.org/wiki/Entertainment|https://en.wikipedia.org/wiki/Humour") == "Humour"
assert extract_topic_labels("https://en.wikipedia.org/wiki/Military|https://en.wikipedia.org/wiki/Society") == "Politics"
assert extract_topic_labels("https://en.wikipedia.org/wiki/Business|https://en.wikipedia.org/wiki/Knowledge") == "Knowledge"
assert not df["topic_categories"].dropna().str.contains("https://", regex=False).any()

MODEL_TOPIC_LABELS = (
    "Entertainment", "Fashion", "Food", "Gaming", "Health", "Hobby",
    "Humour", "Knowledge", "Lifestyle", "Music", "Pet", "Politics",
    "Religion", "Society", "Sports", "Technology", "Tourism", "Vehicle",
)
topic_sets = df["topic_categories"].fillna("").map(
    lambda value: {topic for topic in value.split("|") if topic}
)
TOPIC_FEATURE_COLUMNS = []
for topic in MODEL_TOPIC_LABELS:
    feature = "topic_" + topic.casefold().replace(" ", "_")
    df[feature] = topic_sets.map(lambda values, topic=topic: topic in values)
    TOPIC_FEATURE_COLUMNS.append(feature)
df["topic_missing"] = df["topic_categories"].isna()
TOPIC_FEATURE_COLUMNS.append("topic_missing")
assert df[TOPIC_FEATURE_COLUMNS].dtypes.map(lambda dtype: dtype == bool).all()
assert (df[TOPIC_FEATURE_COLUMNS].sum(axis=1) >= 1).all()
print(df["publish_time_bucket"].value_counts(dropna=False).sort_index())
print("Subscriber tiers:")
print(df["subscriber_tier"].value_counts(dropna=False).reindex(SUBSCRIBER_TIER_ORDER, fill_value=0))
print(
    "Clean channel-average coverage:",
    f"{df['ch_avg_views_per_video_at_publish'].notna().sum():,}/{len(df):,}",
)
print("Topic examples:", df["topic_categories"].dropna().head(3).tolist())
print(f"Created {len(TOPIC_FEATURE_COLUMNS)} binary topic features")

publish_time_bucket
early_morning         4531
evening              26131
late_night            7779
morning_afternoon    26074
Name: count, dtype: Int64
Subscriber tiers:
subscriber_tier
under_1k         5120
1k_to_10k       12510
10k_to_100k     13745
100k_to_250k     4495
250k_to_500k     2705
500k_to_1m       5762
1m_plus         20178
missing             0
Name: count, dtype: Int64
Clean channel-average coverage: 27,511/64,515
Topic examples: ['Politics', 'Politics', 'Politics']
Created 19 binary topic features


In [3]:
HORIZON_COLUMNS = {
    f"d{horizon}_{suffix}"
    for horizon in HORIZONS
    for suffix in ("views", "likes", "comments", "hours_off", "usable")
}
POST_PUBLICATION_AUDIT_COLUMNS = {
    "ch_stats_as_of", "title_changed", "description_changed"
}
EXCLUDED_MODEL_COLUMNS = {
    "video_id", "channel_id", "category_id", "title", "published_at",
    "thumbnail_url", "default_audio_language",
    "eligible", "is_live_broadcast", "channel_stats_backfilled",
    "channel_country",
    "publish_hour_sin", "publish_hour_cos",
    "publish_dow_sin", "publish_dow_cos",
    "title_length", "title_word_count",
    "title_has_number", "title_has_question", "title_has_exclaim",
    "title_upper_ratio", "title_script",
    "ch_views_at_publish",
    "tags",
    "topic_categories",
}


def is_true(series: pd.Series) -> pd.Series:
    """Normalise booleans whether pandas reads them as bool or text."""
    return series.astype("string").str.strip().str.lower().eq("true")


def isolate_horizon(frame: pd.DataFrame, horizon: int) -> pd.DataFrame:
    target = f"d{horizon}_views"
    usable = f"d{horizon}_usable"
    required = {"eligible", target, usable}
    missing = required.difference(frame.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    keep = is_true(frame["eligible"]) & is_true(frame[usable]) & frame[target].notna()
    excluded = (
        (HORIZON_COLUMNS - {target})
        | POST_PUBLICATION_AUDIT_COLUMNS
        | EXCLUDED_MODEL_COLUMNS
    )
    output_columns = [column for column in frame.columns if column not in excluded]
    return frame.loc[keep, output_columns].copy()

In [4]:
horizon_datasets = {}
summary_rows = []

for horizon in HORIZONS:
    target = f"d{horizon}_views"
    usable = f"d{horizon}_usable"
    horizon_df = isolate_horizon(df, horizon)
    output_path = output_dir / f"viewcastlk_day_{horizon}.csv"
    horizon_df.to_csv(output_path, index=False)

    # Read the saved file back so the tests validate the actual CSV, not only memory.
    saved_df = pd.read_csv(output_path, low_memory=False)
    expected_mask = is_true(df["eligible"]) & is_true(df[usable]) & df[target].notna()

    assert len(saved_df) == int(expected_mask.sum())
    assert saved_df[target].notna().all()
    expected_columns = [
        column for column in df.columns
        if column not in (
            (HORIZON_COLUMNS - {target})
            | POST_PUBLICATION_AUDIT_COLUMNS
            | EXCLUDED_MODEL_COLUMNS
        )
    ]
    assert list(saved_df.columns) == expected_columns
    assert set(saved_df.columns).intersection(HORIZON_COLUMNS) == {target}
    assert set(saved_df.columns).isdisjoint(POST_PUBLICATION_AUDIT_COLUMNS)
    assert set(saved_df.columns).isdisjoint(EXCLUDED_MODEL_COLUMNS)
    assert "publish_time_bucket" in saved_df.columns
    assert saved_df["publish_time_bucket"].tolist() == df.loc[expected_mask, "publish_time_bucket"].tolist()
    assert "subscriber_tier" in saved_df.columns
    assert saved_df["subscriber_tier"].notna().all()
    assert set(saved_df["subscriber_tier"]).issubset(SUBSCRIBER_TIER_ORDER)
    assert saved_df["subscriber_tier"].tolist() == df.loc[expected_mask, "subscriber_tier"].tolist()
    assert "ch_avg_views_per_video_at_publish" in saved_df.columns
    expected_average = df.loc[expected_mask, "ch_avg_views_per_video_at_publish"].reset_index(drop=True)
    actual_average = pd.to_numeric(saved_df["ch_avg_views_per_video_at_publish"], errors="coerce")
    pd.testing.assert_series_equal(actual_average, expected_average, check_names=False)

    horizon_datasets[horizon] = horizon_df
    summary_rows.append({
        "horizon": f"day_{horizon}",
        "target": target,
        "saved_rows": len(saved_df),
        "columns": len(saved_df.columns),
        "only_horizon_column": sorted(set(saved_df.columns).intersection(HORIZON_COLUMNS))[0],
        "missing_target": int(saved_df[target].isna().sum()),
        "clean_channel_average_rows": int(saved_df["ch_avg_views_per_video_at_publish"].notna().sum()),
        "clean_channel_average_percent": 100 * saved_df["ch_avg_views_per_video_at_publish"].notna().mean(),
        "file_size_mb": output_path.stat().st_size / 1_000_000,
        "csv_file": str(output_path.relative_to(project_root)),
        "validation": "PASS",
    })

isolation_summary = pd.DataFrame(summary_rows).set_index("horizon")
display(isolation_summary.style.format({
    "file_size_mb": "{:.2f}",
    "clean_channel_average_percent": "{:.2f}%",
}))

subscriber_tier_distribution = pd.concat([
    horizon_datasets[horizon]["subscriber_tier"]
    .value_counts()
    .reindex(SUBSCRIBER_TIER_ORDER, fill_value=0)
    .rename_axis("subscriber_tier")
    .reset_index(name="video_rows")
    .assign(horizon_days=horizon)
    for horizon in HORIZONS
], ignore_index=True)
subscriber_tier_distribution["percent"] = (
    100
    * subscriber_tier_distribution["video_rows"]
    / subscriber_tier_distribution.groupby("horizon_days")["video_rows"].transform("sum")
)
display(subscriber_tier_distribution.pivot(
    index="subscriber_tier", columns="horizon_days", values="percent"
).reindex(SUBSCRIBER_TIER_ORDER).round(2))

horizon_days        7      14     21     30
subscriber_tier                            
under_1k          6.91   7.22   7.66   7.73
1k_to_10k        15.21  17.68  20.26  19.03
10k_to_100k       16.4  19.54  20.75   21.4
100k_to_250k      6.09   6.38   6.95   6.98
250k_to_500k      4.36   5.41   5.13   5.26
500k_to_1m       10.81    9.3   8.95   9.17
1m_plus          40.23  34.48  30.29  30.43
missing            0.0    0.0    0.0    0.0


## Checkpoint

The four CSV files now exist in `Dataset/model_horizon_datasets/`. Each contains the shared `publish_time_bucket`, raw subscriber count, refined `subscriber_tier`, and leakage-safe `ch_avg_views_per_video_at_publish`; no raw `published_at` or raw cumulative channel views; exactly one horizon column—its own view target—and none of the other excluded fields. All assertions must pass before moving on.